# K-Means — analiza klastrów krok po kroku

**Cel szkoleniowy:** pokazać uczenie bez nadzoru i praktyczną analizę skupień.

W notebooku:
1. użyjemy danych Wine,
2. wystandaryzujemy cechy,
3. wybierzemy liczbę klastrów,
4. zastosujemy K-Means,
5. zwizualizujemy klastry przez PCA,
6. zbudujemy profile klastrów,
7. sprawdzimy wpływ parametru `k`.

## 1. Intuicja

K-Means dzieli obserwacje na **k klastrów**.

Algorytm powtarza:
1. przypisanie punktów do najbliższego centroidu,
2. przesunięcie centroidów do środka przypisanych punktów.

Minimalizowana jest suma kwadratów odległości punktów od ich centroidów.

Ważne:
- K-Means działa na odległościach,
- dlatego **skalowanie cech jest bardzo istotne**,
- liczba klastrów `k` musi zostać podana z góry.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42

## 2. Dane

Użyjemy zbioru **Wine**.

Dane zawierają 13 cech chemicznych opisujących próbki wina.  
Etykiety klas istnieją, ale w analizie klastrów **nie używamy ich do trenowania modelu**.

In [ ]:
data = load_wine(as_frame=True)

X = data.data
true_labels = data.target

print("Rozmiar:", X.shape)
X.head()

## 3. Dlaczego skalowanie jest konieczne?

Jeżeli jedna cecha ma zakres 0–1, a druga 0–1000, druga może całkowicie zdominować odległość euklidesową.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame(X_scaled, columns=X.columns).describe().round(2)

## 4. Jak wybrać liczbę klastrów?

Pokażemy dwie popularne metody:

- **elbow method** — szukamy „łokcia” na wykresie inercji,
- **silhouette score** — im wyżej, tym lepiej rozdzielone klastry.

In [ ]:
inertias = []
silhouettes = []
ks = range(2, 9)

for k in ks:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=20
    )
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(7, 5))
plt.plot(list(ks), inertias, marker="o")
plt.xlabel("Liczba klastrów k")
plt.ylabel("Inertia")
plt.title("Elbow method")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(list(ks), silhouettes, marker="o")
plt.xlabel("Liczba klastrów k")
plt.ylabel("Silhouette score")
plt.title("Silhouette score dla różnych k")
plt.show()

pd.DataFrame({
    "k": list(ks),
    "inertia": inertias,
    "silhouette": silhouettes
})

## 5. Model K-Means dla k = 3

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=RANDOM_STATE,
    n_init=20
)

clusters = kmeans.fit_predict(X_scaled)

cluster_counts = pd.Series(clusters).value_counts().sort_index()
cluster_counts

## 6. Wizualizacja klastrów przez PCA

Ponieważ mamy 13 cech, redukujemy wymiar do 2 składowych PCA tylko na potrzeby wizualizacji.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "cluster": clusters
})

plt.figure(figsize=(8, 6))

for cluster_id in sorted(plot_df["cluster"].unique()):
    subset = plot_df[plot_df["cluster"] == cluster_id]
    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=f"Klaster {cluster_id}",
        alpha=0.75
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-Means — klastry po redukcji PCA")
plt.legend()
plt.show()

print("Wyjaśniona wariancja przez 2 PC:",
      round(pca.explained_variance_ratio_.sum(), 3))

## 7. Profilowanie klastrów

Najważniejszy element biznesowej analizy klastrów:  
**nie tylko znaleźć klastry, ale zrozumieć czym się różnią.**

In [ ]:
df_profile = X.copy()
df_profile["cluster"] = clusters

cluster_profiles = df_profile.groupby("cluster").mean().round(2)
cluster_profiles

In [ ]:
global_mean = X.mean()
profile_relative = cluster_profiles / global_mean

profile_relative.T.plot(kind="bar", figsize=(14, 6))
plt.axhline(1.0)
plt.ylabel("Średnia klastra / średnia globalna")
plt.title("Profil klastrów względem średniej globalnej")
plt.legend(title="Klaster")
plt.show()

## 8. Porównanie z prawdziwymi klasami — tylko dydaktycznie

To nie jest element uczenia K-Means.  
Używamy prawdziwych etykiet tylko po to, by zobaczyć, czy naturalne grupy w danych przypominają klasy eksperckie.

In [ ]:
comparison = pd.crosstab(
    pd.Series(true_labels, name="Prawdziwa klasa"),
    pd.Series(clusters, name="Klaster K-Means")
)

comparison

## 9. Eksperyment: wpływ k

Spróbuj:
- `k=2`,
- `k=3`,
- `k=4`,
- `k=5`.

Porównaj:
- silhouette score,
- liczebność klastrów,
- profile średnich wartości cech.

In [ ]:
summary = []

for k in range(2, 7):
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=20
    )
    labels = model.fit_predict(X_scaled)

    summary.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "najmniejszy_klaster": np.bincount(labels).min(),
        "najwiekszy_klaster": np.bincount(labels).max()
    })

pd.DataFrame(summary)

## 10. Opcje do pokazania na szkoleniu

Można zmieniać:
- `n_clusters`,
- `n_init`,
- `max_iter`,
- `random_state`.

Można też pokazać różnicę między:
- K-Means **bez skalowania**,
- K-Means **po StandardScaler**.

### Ćwiczenie
1. Uruchom K-Means bez `StandardScaler`.
2. Porównaj silhouette score.
3. Zmień `k` z 2 do 6.
4. Wybierz własną najlepszą liczbę klastrów.
5. Nazwij klastry na podstawie profili cech.

### Wniosek
K-Means jest prosty i bardzo użyteczny, ale trzeba pamiętać:
- wynik zależy od skali cech,
- trzeba dobrać `k`,
- klastry wymagają interpretacji,
- nie każda struktura danych ma kształt odpowiedni dla K-Means.